In [5]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# torch.multiprocessing.set_start_method('spawn', force=True)
import time
import importlib
import torch.optim as optim
from torchsummary import summary
import os
import statistics

os.chdir("../")
from src.data_preparation import SimulatedDataset, collate_function, load_real_data
from src.transformers import MiniTransformer
import src.transformers as transformerFunctions
from src.transformers import init_weights_recursive
from src.transformers import print_parameters
from src.transformers import create_custom_mask_pair,create_custom_mask_pred, create_distance_to_end_matrix, create_pairwise_distance_matrix
from src.evaluation import calculate_bench1_loss, calculate_bench2_loss, calculate_repeat_loss, calculate_regression_loss, evaluate_mini_transformer
from src.statistical_testing import statistical_testing, print_p_values, plot_context_predindex_pair_effect, get_context_predindex_pair_effect
import torch.autograd.profiler as profiler
from sklearn.model_selection import KFold
device = torch.device("cpu")

In [6]:
data_str = "pbc2"
data, maxlen = load_real_data(data_str)

FileNotFoundError: [Errno 2] No such file or directory: 'data/pbc2/pbc2_binarised.csv'

In [ ]:

# get number of sequences
print(f"Number of sequences: {len(data)}")


# get min, max, median of sequence length
seq_lengths = [len(seq) for seq in data]
print(f"Minimum sequence length: {min(seq_lengths)}")
print(f"Maximum sequence length: {max(seq_lengths)}")
print(f"Median sequence length: {statistics.median(seq_lengths)}")



Number of sequences: 252
Minimum sequence length: 3
Maximum sequence length: 10
Median sequence length: 6.5


In [ ]:

if __name__ == '__main__':
    
    bench_repeat_loss_list = []
    regression_loss_total_list = []
    model_loss_total_list = []
    regression_loss_predindex_list = []
    model_loss_predindex_list = []
    bench1_loss_list = []
    bench2_loss_list = []
    bench2loss_predindex_list = []
    bench_repeat_loss_predindex_list = []
    benchloss_predindex_list = []
    models = []
    


    # Hyperparameters
    data_str = "pbc2"
    # data_str = "ghq_b_sum"
    # data_str = "ghq_sum"
    # data_str = "simulation"
    batch_size = 1          # Batch size for loading data
    dk = 1                  # d_k
    dv = 1                  # d_v
    nheads = 4             # number of heads
    ncum = 4                 # number of cumulants
    maxlen = 10             # maximum length of the sequence
    learning_rate = 5e-4
    lambda_l2 = 1e-3
    EPOCHS = 200
    target_sample_size = 8
    nrepp = 10
    seeds = [0, 1, 11, 42, 123, 999, 1337, 2025, 9999, 12345]
    
    k_for_cross_val = 10
    
    # load real data
    data, maxlen = load_real_data(data_str)
    
    
    # Create the KFold splitter:
    kf = KFold(n_splits=k_for_cross_val, shuffle=True, random_state=42)

    # Build a list of (train, eval) folds:
    folds = []
    for train_index, eval_index in kf.split(data):
        train_data = [data[i] for i in train_index]
        eval_data  = [data[i] for i in eval_index]
        folds.append((train_data, eval_data))
    
    
    cnt = 0
    for (train, val) in folds:
        
        # Set the random seed for reproducibility
        torch.manual_seed(seeds[cnt])
        cnt += 1


        
        n = len(train)
        p = train[0].shape[1]
        print("Train size: ", len(train), "\n")
        print("Val size: ", len(val), "\n")
        
        train_dataset = train
        eval_dataset = val
        predindex = 9

        
        mask_pairwise = create_custom_mask_pair(maxlen, device)
        distance_to_end_matrix = create_distance_to_end_matrix(maxlen, device)
        pairwise_distance_matrix = create_pairwise_distance_matrix(maxlen, device)

    
        dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_function,  num_workers=0)
        eval_dataloader = DataLoader(eval_dataset, batch_size=len(eval_dataset), shuffle=False, collate_fn=collate_function, num_workers=0)
       
        model = MiniTransformer(p, nheads, dk, dv, ncum, mask_pairwise, pairwise_distance_matrix, distance_to_end_matrix,  device)

        # model.apply(init_weights_recursive)
        model.to(device)

        # model = torch.compile(model)
        # Start the timer
        start_time = time.time()

        # Define optimizer
        optimizer = optim.Adam(model.parameters(), lr= learning_rate, weight_decay=lambda_l2)
        # optimizer = optim.Adam(model.parameters(), lr= learning_rate)
        
        print("Number of Parameters", transformerFunctions.count_parameters(model))
        
        run_path = transformerFunctions.train_mini_transformer(model, dataloader, eval_dataloader, optimizer, lambda_l2, EPOCHS, device)


        # # Enable profiling
        # with profiler.profile() as prof:

        #     model.eval() 
        #     # Forward pass
        #     output = model(train_dataset.data)
        #     # Backward pass (this will profile the backward pass as well)
        #     output.backward(torch.ones_like(output))

        # # Print profiling results
        # print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))



        # End the timer
        end_time = time.time()

        # Calculate and print the execution time
        execution_time = end_time - start_time 
        print(f"Execution time: {execution_time:.6f} seconds")


        
        
        
        # Evaluate the model
        
        eval_dataloader = DataLoader(eval_dataset, batch_size=1, shuffle=False, collate_fn=collate_function, num_workers=0)
       
    
    
        dimave, bench1_loss, benchloss_predindex = calculate_bench1_loss(train_dataset, eval_dataset, predindex)
        regression_loss_predindex, regression_loss_total = calculate_regression_loss(train_dataset, eval_dataset, predindex)
        model_loss_predindex, model_loss_total = evaluate_mini_transformer(eval_dataloader, model, predindex)
        
        
        print("baseline average loss: ", bench1_loss)

        
        # Evaluate the model
        if data_str == "simulation":
            bench2loss, bench2loss_predindex = calculate_bench2_loss(train_dataset, eval_dataset, dimave)
            print("baseline informed loss: ", bench2loss)
            bench2_loss_list.append(bench2loss)
            bench2loss_predindex_list.append(bench2loss_predindex)
            
        else:
            bench_repeat, bench_repeat_predindex = calculate_repeat_loss(eval_dataset, predindex)    
            print("baseline repeat: ", bench_repeat)
            bench_repeat_loss_list.append(bench_repeat)
            
            
        print("regression loss total: ", regression_loss_total)
        print("model loss total: ", model_loss_total.item(), "\n") 
        
        
        if data_str == "simulation":
            print("baseline informed loss predindex: ", bench2loss_predindex)
            
            
        print("baseline average loss predindex: ", benchloss_predindex)
        print("baseline repeat predindex: ", bench_repeat_predindex)   
        print("regression loss predindex: ", regression_loss_predindex)
        print("model loss predindex: ", model_loss_predindex.item()) 

        
        bench_repeat_loss_list.append(bench_repeat)
        regression_loss_total_list.append(regression_loss_total)
        model_loss_total_list.append(model_loss_total.item())
        bench1_loss_list.append(bench1_loss)
        
        benchloss_predindex_list.append(benchloss_predindex)
        bench_repeat_loss_predindex_list.append(bench_repeat_predindex)
        regression_loss_predindex_list.append(regression_loss_predindex)
        model_loss_predindex_list.append(model_loss_predindex.item())
        
        
        
        
# save the mean and std of these lists in text file in the format of mean ± std 
# os.chdir("./notebooks/results")




with open(f"./notebooks/pred_{data_str}_results_n={n}_batch_size={batch_size}_p={p}_ncum={ncum}_nheads={nheads}_epochs={EPOCHS}.txt", "a") as f:
    # First line
    f.write(f"{data_str} results n = {n}\n")

    # Single-line writes with three-decimal formatting:
    f.write(f"baseline repeat: {statistics.mean(bench_repeat_loss_list):.3f} ± {statistics.stdev(bench_repeat_loss_list):.3f}\n")
    f.write(f"baseline average loss: {statistics.mean(bench1_loss_list):.3f} ± {statistics.stdev(bench1_loss_list):.3f}\n")
    if data_str == "simulation":
        f.write(f"baseline informed loss: {statistics.mean(bench2_loss_list):.3f} ± {statistics.stdev(bench2_loss_list):.3f}\n")
    f.write(f"regression loss total: {statistics.mean(regression_loss_total_list):.3f} ± {statistics.stdev(regression_loss_total_list):.3f}\n")
    f.write(f"model loss total: {statistics.mean(model_loss_total_list):.3f} ± {statistics.stdev(model_loss_total_list):.3f}\n\n")

    
    f.write(f"baseline repeat predindex: {statistics.mean(bench_repeat_loss_predindex_list):.3f} ± {statistics.stdev(bench_repeat_loss_list):.3f}\n")
    f.write(f"baseline average loss predindex: {statistics.mean(benchloss_predindex_list):.3f} ± {statistics.stdev(benchloss_predindex_list):.3f}\n")
  
    f.write(f"regression loss predindex: {statistics.mean(regression_loss_predindex_list):.3f} ± {statistics.stdev(regression_loss_predindex_list):.3f}\n")
    if data_str == "simulation":
        f.write(f"baseline informed loss predindex: {statistics.mean(bench2loss_predindex_list):.3f} ± {statistics.stdev(bench2loss_predindex_list):.3f}\n")
    f.write(f"model loss predindex: {statistics.mean(model_loss_predindex_list):.3f} ± {statistics.stdev(model_loss_predindex_list):.3f}\n")

    # Extra newlines at the end
    f.write("\n\n")


Train size:  226 

Val size:  26 

Number of Parameters 564
EPOCH 1:
avg_loss:     15.08680 penalty    : 0.01771
avg_loss_val: 8.49954 penalty_val: 0.01771
EPOCH 11:
avg_loss:     6.09253 penalty    : 0.02238
avg_loss_val: 5.63935 penalty_val: 0.02239
EPOCH 21:
avg_loss:     5.74552 penalty    : 0.02606
avg_loss_val: 5.25192 penalty_val: 0.02607
EPOCH 31:
avg_loss:     5.66271 penalty    : 0.02880
avg_loss_val: 5.17493 penalty_val: 0.02880
EPOCH 41:
avg_loss:     5.62616 penalty    : 0.03096
avg_loss_val: 5.14264 penalty_val: 0.03096
EPOCH 51:
avg_loss:     5.59375 penalty    : 0.03262
avg_loss_val: 5.09052 penalty_val: 0.03262
EPOCH 61:
avg_loss:     5.56407 penalty    : 0.03394
avg_loss_val: 5.04089 penalty_val: 0.03394
EPOCH 71:
avg_loss:     5.53945 penalty    : 0.03484
avg_loss_val: 5.00216 penalty_val: 0.03484
EPOCH 81:
avg_loss:     5.52786 penalty    : 0.03570
avg_loss_val: 4.96623 penalty_val: 0.03570
EPOCH 91:
avg_loss:     5.51888 penalty    : 0.03634
avg_loss_val: 4.94002 p